In [ ]:
# Cell 1: project setup. Re-uses the path setup from 00_colab_setup.ipynb via the shared
# helper project_setup.py (no path/mount logic is duplicated here).
import sys
from pathlib import Path

def _find_notebooks_dir():
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base, base / "notebooks", base / "ml" / "notebooks"):
            if (cand / "project_setup.py").is_file():
                return cand
    return None

_nb_dir = _find_notebooks_dir()
if _nb_dir is None:
    # Fresh Colab runtime: Drive is not mounted yet, so the helper (stored on Drive) can't be seen.
    try:
        from google.colab import drive
    except ImportError as err:
        raise FileNotFoundError(
            "Could not find ml/notebooks/project_setup.py. Start Jupyter from inside the project "
            "folder (see ml/README.md)."
        ) from err
    drive.mount("/content/drive")
    _hits = sorted(Path("/content/drive/MyDrive").glob("*/ml/notebooks/project_setup.py"))
    _hits += sorted(Path("/content/drive/MyDrive").glob("*/*/ml/notebooks/project_setup.py"))
    if not _hits:
        raise FileNotFoundError(
            "project_setup.py not found on your Drive. Upload the whole crop-disease-detection "
            "folder to Drive (see 00_colab_setup.ipynb)."
        )
    _nb_dir = _hits[0].parent
sys.path.insert(0, str(_nb_dir))

from project_setup import setup_project
paths = setup_project()
RAW_DIR, EVAL_DIR = paths.raw_dir, paths.eval_dir

if not RAW_DIR.is_dir() or not any(p.is_dir() for p in RAW_DIR.iterdir()):
    raise FileNotFoundError(
        f"No class folders found in {RAW_DIR}\n"
        "Download the dataset first:  python ml/dataset/download_dataset.py"
    )
print("Raw dataset folder:", RAW_DIR)

## 1. Image counts per class

Counts `.jpg` / `.jpeg` / `.png` files (case-insensitive) in every class folder of `ml/dataset/raw/`, which is filled by `ml/dataset/download_dataset.py`. Empty classes are highlighted in red.

In [ ]:
# Prompt 7: count images per class folder (case-insensitive .jpg / .jpeg / .png)
import os
import pandas as pd
from IPython.display import display

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

class_dirs = sorted(
    p for p in RAW_DIR.iterdir()
    if p.is_dir() and not p.name.startswith(".") and p.name != "__MACOSX"
)

rows, nested = [], []
for d in class_dirs:
    n_images = sum(
        1 for f in d.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS
    )
    if any(x.is_dir() for x in d.iterdir()):
        nested.append(d.name)
    rows.append({"class_name": d.name, "image_count": n_images})

df = (
    pd.DataFrame(rows, columns=["class_name", "image_count"])
    .sort_values("image_count", ascending=False, kind="stable")
    .reset_index(drop=True)
)

# Everything on disk (for reconciling with: find ml/dataset/raw -type f | wc -l)
files_on_disk = sum(len(files) for _, _, files in os.walk(RAW_DIR))
total_images = int(df["image_count"].sum())
empty_classes = df.loc[df["image_count"] == 0, "class_name"].tolist()
zero = set(empty_classes)


def _flag_empty(row):
    red = "background-color: #f8c9c9; color: #8a1f1f; font-weight: bold"
    return [red if row["class_name"] in zero else "" for _ in row]


try:
    styled = (
        df.style
        .background_gradient(subset=["image_count"], cmap="Greens")
        .apply(_flag_empty, axis=1)
        .format({"class_name": lambda s: f"\u26a0 {s}  (EMPTY - 0 images)" if s in zero else s})
        .hide(axis="index")
    )
    display(styled)
except ImportError:  # jinja2 missing -> plain table, still flag empties
    plain = df.copy()
    plain["status"] = ["EMPTY - 0 images" if c == 0 else "" for c in plain["image_count"]]
    display(plain)

if empty_classes:
    print(f"\u26a0\ufe0f  WARNING: {len(empty_classes)} class folder(s) contain NO images "
          f"(failed or partial download?): {', '.join(empty_classes)}")
    print("   Fix: python ml/dataset/download_dataset.py --force")
if nested:
    print(f"\u26a0\ufe0f  WARNING: these class folders contain sub-folders, whose images are NOT "
          f"counted: {', '.join(nested)}")

print(f"\nFiles on disk: {files_on_disk}  |  images counted: {total_images}  |  "
      f"non-image / nested files ignored: {files_on_disk - total_images}")
print(f"TOTAL: {total_images} images across {len(df)} classes "
      f"({len(empty_classes)} empty)")

## 2. Class distribution chart

Reuses the table above and saves the figure to `ml/evaluation/class_distribution.png`.

In [ ]:
# Prompt 8: class-distribution bar chart (uses the DataFrame `df` from the cell above; no re-count)
import matplotlib.pyplot as plt

if "df" not in globals() or df.empty:
    raise RuntimeError("Run the counting cell above first: `df` is missing or empty.")

plot_df = df.sort_values("image_count", ascending=False, kind="stable")  # descending
horizontal = len(plot_df) > 15
BAR_COLOR, EMPTY_COLOR = "#2f7d4f", "#c0392b"
colors = [EMPTY_COLOR if c == 0 else BAR_COLOR for c in plot_df["image_count"]]

if horizontal:
    fig, ax = plt.subplots(figsize=(11, max(4.0, 0.38 * len(plot_df) + 1.5)))
    bars = ax.barh(plot_df["class_name"], plot_df["image_count"], color=colors)
    ax.invert_yaxis()                       # largest bar at the top
    ax.set_xlabel("Number of Images")
    ax.set_ylabel("Class")
    ax.set_xlim(0, plot_df["image_count"].max() * 1.12 or 1)
    ax.grid(axis="x", alpha=0.25)
else:
    fig, ax = plt.subplots(figsize=(max(7.0, 0.9 * len(plot_df) + 2), 6))
    bars = ax.bar(plot_df["class_name"], plot_df["image_count"], color=colors)
    ax.set_ylabel("Number of Images")
    ax.set_xlabel("Class")
    ax.set_ylim(0, plot_df["image_count"].max() * 1.12 or 1)
    ax.grid(axis="y", alpha=0.25)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

ax.bar_label(bars, padding=3, fontsize=8)  # exact count at the end of each bar
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.set_title("PlantVillage Dataset \u2014 Class Distribution")

EVAL_DIR.mkdir(parents=True, exist_ok=True)
out_path = EVAL_DIR / "class_distribution.png"
fig.savefig(out_path, dpi=200, bbox_inches="tight")  # overwrites any previous file
print(f"Saved {out_path} ({out_path.stat().st_size / 1024:.0f} KB)")
plt.show()